In [9]:
import mysql.connector
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# ---------------------------------------------------
# MYSQL CONNECTION
# ---------------------------------------------------

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="sql09",
    database="ecommerce_anomaly_db"
)

print("MySQL Connected Successfully")

# ---------------------------------------------------
# OUTPUT FOLDER
# ---------------------------------------------------

output_folder = r"C:\Users\naman\Downloads\output\dashboard_outputs"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# ---------------------------------------------------
# LOAD DATA
# ---------------------------------------------------

query = "SELECT * FROM anomaly_alerts"

df = pd.read_sql(query, conn)

print(f"Total Rows Loaded: {len(df)}")

# ---------------------------------------------------
# GLOBAL SETTINGS
# ---------------------------------------------------

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# ---------------------------------------------------
# 1. RISK LEVEL DISTRIBUTION
# ---------------------------------------------------

risk_counts = df['risk_level'].value_counts()

risk_colors = {
    'LOW': 'green',
    'MEDIUM': 'orange',
    'HIGH': 'red'
}

colors = [
    risk_colors.get(level, 'gray')
    for level in risk_counts.index
]

plt.figure()

bars = plt.bar(
    risk_counts.index,
    risk_counts.values,
    color=colors
)

plt.title(
    "Risk Level Distribution",
    fontsize=15,
    fontweight='bold'
)

plt.xlabel("Risk Level")
plt.ylabel("Total Alerts")
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        int(bar.get_height()),
        ha='center'
    )

plt.tight_layout()

plt.savefig(
    f"{output_folder}/1_risk_level_distribution.png",
    dpi=300
)

plt.close()

# ---------------------------------------------------
# 2. TOP 20 SUSPICIOUS USERS
# ---------------------------------------------------

top_users = (
    df['user_id']
    .value_counts()
    .head(20)
)

plt.figure()

bars = plt.barh(
    top_users.index.astype(str),
    top_users.values,
    color='steelblue'
)

plt.title(
    "Top 20 Suspicious Users",
    fontsize=15,
    fontweight='bold'
)

plt.xlabel("Alert Count")
plt.ylabel("User ID")

plt.grid(axis='x', alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/2_top_suspicious_users.png",
    dpi=300
)

plt.close()

# ---------------------------------------------------
# 3. TOP ANOMALY REASONS
# ---------------------------------------------------

reason_counts = (
    df['reason']
    .value_counts()
    .head(10)
)

colors = plt.cm.Set3(
    np.linspace(
        0,
        1,
        len(reason_counts)
    )
)

plt.figure(figsize=(18, 12))

bars = plt.bar(
    reason_counts.index,
    reason_counts.values,
    color=colors
)

plt.title(
    "Top Anomaly Reasons",
    fontsize=15,
    fontweight='bold'
)

plt.xlabel("Anomaly Reason")
plt.ylabel("Frequency")

plt.xticks(rotation=85)

plt.grid(axis='y', alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/3_top_anomaly_reasons.png",
    dpi=300
)

plt.close()

# ---------------------------------------------------
# 4. ANOMALY TREND OVER TIME
# ---------------------------------------------------

df['event_timestamp'] = pd.to_datetime(
    df['event_timestamp']
)

df['hour'] = (
    df['event_timestamp']
    .dt.hour
)

hourly_alerts = (
    df.groupby('hour')
    .size()
)

plt.figure()

plt.plot(
    hourly_alerts.index,
    hourly_alerts.values,
    marker='o',
    linewidth=2
)

plt.title(
    "Anomaly Trend by Hour",
    fontsize=15,
    fontweight='bold'
)

plt.xlabel("Hour of Day")
plt.ylabel("Alert Count")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/4_hourly_trend.png",
    dpi=300
)

plt.close()

# ---------------------------------------------------
# 5. RISK SCORE DISTRIBUTION
# ---------------------------------------------------

plt.figure()

plt.hist(
    df['risk_score'],
    bins=30,
    alpha=0.8
)

plt.title(
    "Risk Score Distribution",
    fontsize=15,
    fontweight='bold'
)

plt.xlabel("Risk Score")
plt.ylabel("Frequency")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/5_risk_score_distribution.png",
    dpi=300
)

plt.close()

# ---------------------------------------------------
# 6. HIGHEST RISK USERS
# ---------------------------------------------------

highest_risk = (

    df.groupby('user_id')
    ['risk_score']
    .max()
    .sort_values(
        ascending=False
    )
    .head(20)

)

plt.figure()

plt.bar(
    highest_risk.index.astype(str),
    highest_risk.values,
    color='crimson'
)

plt.title(
    "Highest Risk Users",
    fontsize=15,
    fontweight='bold'
)

plt.xlabel("User ID")
plt.ylabel("Max Risk Score")

plt.xticks(rotation=45)

plt.grid(axis='y', alpha=0.3)

plt.tight_layout()

plt.savefig(
    f"{output_folder}/6_highest_risk_users.png",
    dpi=300
)

plt.close()

# ---------------------------------------------------
# 7. KPI SUMMARY IMAGE
# ---------------------------------------------------

total_alerts = len(df)

avg_score = round(
    df['risk_score'].mean(),
    2
)

top_reason = (
    df['reason']
    .value_counts()
    .idxmax()
)

highest_user = (

    df.groupby('user_id')
    ['risk_score']
    .max()
    .idxmax()

)

plt.figure(figsize=(10, 5))

plt.axis('off')

summary_text = f"""

REAL-TIME E-COMMERCE ANOMALY DETECTION

Total Alerts: {total_alerts}

Average Risk Score: {avg_score}

Top Anomaly Reason:
{top_reason}

Highest Risk User:
{highest_user}

"""

plt.text(
    0.5,
    0.5,
    summary_text,
    ha='center',
    va='center',
    fontsize=16,
    fontweight='bold'
)

plt.savefig(
    f"{output_folder}/7_kpi_summary.png",
    dpi=300,
    bbox_inches='tight'
)

plt.close()

# ---------------------------------------------------
# CLOSE CONNECTION
# ---------------------------------------------------

conn.close()

print("\nDashboard Generated Successfully!")
print(
    f"Saved in folder:\n{output_folder}"
)

MySQL Connected Successfully


C:\Users\naman\AppData\Local\Temp\ipykernel_27972\2430494040.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Total Rows Loaded: 23604

Dashboard Generated Successfully!
Saved in folder:
C:\Users\naman\Downloads\output\dashboard_outputs
